In [1]:
# Install necessary packages
!pip install requests beautifulsoup4 pandas

You should consider upgrading via the '/Users/samposatama/Documents/rescue-dog-aggregator/venv/bin/python3 -m pip install --upgrade pip' command.


In [17]:
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime
import pandas as pd
from IPython.display import display, HTML

In [18]:
def get_page(url):
    """Get HTML content from a URL with proper headers and error handling"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml',
        'Accept-Language': 'en-US,en;q=0.9',
    }
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise an exception for HTTP errors
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

In [19]:
base_url = "https://www.petsinturkey.org"
dogs_url = f"{base_url}/dogs"

html_content = get_page(dogs_url)

if html_content:
    print(f"Successfully fetched the page. Content length: {len(html_content)} characters")
    # Create BeautifulSoup object
    soup = BeautifulSoup(html_content, 'html.parser')
    # Print the title to confirm correct page
    print(f"Page title: {soup.title.text if soup.title else 'No title'}")
else:
    print("Failed to fetch the page")

Successfully fetched the page. Content length: 1531097 characters
Page title: Adopt a Dog | Pets in Turkey


In [20]:
# Find all dog name elements (formatted as "I'm [Name]")
dog_name_elements = soup.find_all(string=lambda t: t and "I'm " in t)

dog_names = [elem.strip().replace("I'm ", "") for elem in dog_name_elements]
unique_dog_names = set(dog_names)

print(f"Found {len(dog_name_elements)} dog name elements")
print(f"Found {len(unique_dog_names)} unique dog names")
print(f"Dog names: {', '.join(sorted(unique_dog_names))}")

Found 33 dog name elements
Found 33 unique dog names
Dog names: Archie, Arthur, Barney, Chiara, Clark, Coco, Coffee, Doby, Draco, Fındık, Gloria, Grace, Harry, Jack, Jamie, Kali, Lara, Leon, Linda, Luna, Mila, Norman, Oreo, Orky, Rusty, Shadow, Shansli, Sophie, Summer, Tess, Tilley, Whisper, Yuki


In [21]:
# Take the first dog name element
first_dog_name_elem = dog_name_elements[0]
dog_name = first_dog_name_elem.strip().replace("I'm ", "")
print(f"First dog name: {dog_name}")

# Try to find the container for this dog
container = first_dog_name_elem.parent
while container and container.name != 'div':
    container = container.parent

if container:
    print(f"Container ID: {container.get('id', 'No ID')}")
    
    # Extract all text elements in this container
    text_elements = [t.strip() for t in container.find_all(string=True) if t.strip()]
    print(f"Text elements in this container: {text_elements}")
    
    # Try to find status ("Ready to fly")
    status_elements = container.find_all(string=lambda t: t and "Ready to fly" in t)
    if status_elements:
        print(f"Status: {status_elements[0].strip()}")
    else:
        print("No status found")
        
    # Try to find images
    images = container.find_all('img')
    print(f"Found {len(images)} images")
    for i, img in enumerate(images):
        print(f"Image {i+1} src: {img.get('src', 'No src')}")
else:
    print("Couldn't find container for this dog")

First dog name: Norman
Container ID: comp-kwtd833b__item1
Text elements in this container: ["I'm Norman"]
No status found
Found 0 images


In [22]:
def extract_dog_data(name_element):
    """Extract data for a single dog"""
    dog_name = name_element.strip().replace("I'm ", "")
    
    # Find the dog's container
    container = name_element.parent
    while container and container.name != 'div':
        container = container.parent
    
    if not container:
        return None
    
    # Initialize dog data
    dog_data = {
        'name': dog_name,
        'source': 'Pets in Turkey',
    }
    
    # Find the "Ready to fly" status
    ready_elements = container.find_all(string=lambda t: t and "Ready to fly" in t)
    if ready_elements:
        dog_data['status'] = ready_elements[0].strip()
    
    # Find all text in the container to extract data
    all_text = [t.strip() for t in container.find_all(string=True) if t.strip()]
    
    # Look for specific fields
    fields = ['Breed', 'Weight', 'Age', 'Sex', 'Neutered', 'Spayed']
    
    # For each field, find the field label and then try to find the next text element
    for field in fields:
        try:
            field_index = all_text.index(field)
            # The value is likely to be after the field label
            if field_index + 1 < len(all_text):
                value = all_text[field_index + 1]
                if value not in fields:  # Make sure we're not picking up another field
                    dog_data[field.lower()] = value
        except ValueError:
            continue  # Field not found
    
    # Find "Adopt Me" link
    adopt_links = container.find_all('a', string=lambda t: t and "Adopt Me" in t)
    if adopt_links:
        link = adopt_links[0].get('href', '')
        if link and not link.startswith('http'):
            link = base_url + link
        dog_data['adopt_link'] = link
    
    # Find the dog's image
    images = container.find_all('img')
    if images:
        image_urls = []
        for img in images:
            src = img.get('src', '')
            if src and src.lower().endswith(('jpg', 'jpeg', 'png', 'webp', 'gif')):
                if not src.startswith('http'):
                    src = base_url + src
                image_urls.append(src)
        
        if image_urls:
            dog_data['images'] = image_urls
    
    return dog_data

# Test with the first dog
first_dog_data = extract_dog_data(dog_name_elements[0])
print(json.dumps(first_dog_data, indent=2))

{
  "name": "Norman",
  "source": "Pets in Turkey"
}


In [23]:
# Extract data for all dogs
all_dogs_data = []

for name_element in dog_name_elements:
    dog_data = extract_dog_data(name_element)
    if dog_data:
        all_dogs_data.append(dog_data)

print(f"Successfully extracted data for {len(all_dogs_data)} dogs")

# Display the first 3 dogs as a sample
print("\nSample dog data (first 3 dogs):")
for i, dog in enumerate(all_dogs_data[:3]):
    print(f"\nDog {i+1}: {dog['name']}")
    for key, value in dog.items():
        if key != 'name' and key != 'images':  # Skip images to keep output clean
            print(f"  {key}: {value}")
    if 'images' in dog:
        print(f"  images: {len(dog['images'])} images found")

Successfully extracted data for 33 dogs

Sample dog data (first 3 dogs):

Dog 1: Norman
  source: Pets in Turkey

Dog 2: Leon
  source: Pets in Turkey

Dog 3: Orky
  source: Pets in Turkey


In [24]:
# Convert to DataFrame for easier analysis
df = pd.DataFrame(all_dogs_data)

# Basic stats
print(f"Total dogs: {len(df)}")
if 'breed' in df.columns:
    print("\nBreed distribution:")
    display(df['breed'].value_counts())

if 'sex' in df.columns:
    print("\nSex distribution:")
    display(df['sex'].value_counts())

if 'status' in df.columns:
    print("\nStatus distribution:")
    display(df['status'].value_counts())

Total dogs: 33


In [25]:
# Save the extracted data to a JSON file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"../data/pets_in_turkey_dogs_{timestamp}.json"

# Create the data directory if it doesn't exist
import os
os.makedirs("../data", exist_ok=True)

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_dogs_data, f, ensure_ascii=False, indent=2)

print(f"Saved dog data to {output_file}")

Saved dog data to ../data/pets_in_turkey_dogs_20250329_121120.json


In [26]:
def extract_dog_data(name_element):
    """Extract data for a single dog with improved approach"""
    dog_name = name_element.strip().replace("I'm ", "")
    
    # Find the dog's container
    container = name_element.parent
    while container and container.name != 'div':
        container = container.parent
    
    if not container:
        print(f"Couldn't find container for {dog_name}")
        return None
    
    # Initialize dog data
    dog_data = {
        'name': dog_name,
        'source': 'Pets in Turkey',
        'scraped_at': datetime.now().isoformat()
    }
    
    # Find the "Ready to fly" status
    status_elements = [elem.strip() for elem in container.find_all(string=lambda t: t and 
                                                                  ("Ready to fly" in t or 
                                                                  "Currently in" in t))]
    if status_elements:
        dog_data['status'] = status_elements[0]
    
    # Looking at the structure we found earlier, we need to find all text elements
    # and then organize them by field
    
    # Get all text elements from the container
    all_text_elements = container.find_all(string=True)
    all_text = [t.strip() for t in all_text_elements if t.strip()]
    
    # Print the entire text list for debugging (just for the first dog)
    if dog_name == "Norman":  # Only print for the first dog to avoid flooding output
        print("All text elements in container:")
        for i, text in enumerate(all_text):
            print(f"{i}: {text}")
    
    # We know the fields are: Breed, Weight, Age, Sex, Neutered/Spayed
    # These fields appear in sequence in the HTML
    
    # Find indices of field headers
    field_indices = {}
    fields = ['Breed', 'Weight', 'Age', 'Sex', 'Neutered', 'Spayed']
    
    for field in fields:
        try:
            idx = all_text.index(field)
            field_indices[field] = idx
        except ValueError:
            pass
    
    # If we found any field headers, extract their values
    if field_indices:
        # Sort field indices to process them in order
        sorted_fields = sorted(field_indices.items(), key=lambda x: x[1])
        
        for i, (field, idx) in enumerate(sorted_fields):
            # The value is usually one or two elements after the field name
            # Try different offsets
            for offset in [1, 2, 3]:
                if idx + offset < len(all_text):
                    value = all_text[idx + offset]
                    # Check if this isn't another field name
                    if value not in fields:
                        # For height info that's often with weight
                        if field == 'Weight' and 'height' in value.lower():
                            parts = value.split(',')
                            if len(parts) > 1:
                                dog_data['weight'] = parts[0].strip()
                                dog_data['height'] = parts[1].strip()
                            else:
                                dog_data['weight'] = value
                                # Try to find height in the next element
                                if idx + offset + 1 < len(all_text) and 'height' in all_text[idx + offset + 1].lower():
                                    dog_data['height'] = all_text[idx + offset + 1].strip()
                        else:
                            dog_data[field.lower()] = value
                        break
    
    # Find "Adopt Me" link
    adopt_links = container.find_all('a', string="Adopt Me")
    if adopt_links:
        link = adopt_links[0].get('href', '')
        if link and not link.startswith('http'):
            link = base_url + link
        dog_data['adopt_link'] = link
    
    # Find the dog's image
    images = container.find_all('img')
    if images:
        image_urls = []
        for img in images:
            src = img.get('src', '')
            if src and not src.startswith('data:'):  # Exclude base64 encoded images
                if not src.startswith('http'):
                    src = base_url + ('' if src.startswith('/') else '/') + src
                image_urls.append(src)
        
        if image_urls:
            dog_data['images'] = image_urls
    
    return dog_data

In [27]:
# Test with a few dogs for detailed diagnosis
test_dogs = ['Norman', 'Leon', 'Orky']
for dog_name in test_dogs:
    # Find the dog name element
    for elem in dog_name_elements:
        if dog_name in elem:
            print(f"\n--- Detailed extraction for {dog_name} ---")
            dog_data = extract_dog_data(elem)
            print(json.dumps(dog_data, indent=2))
            break


--- Detailed extraction for Norman ---
All text elements in container:
0: I'm Norman
{
  "name": "Norman",
  "source": "Pets in Turkey",
  "scraped_at": "2025-03-29T12:12:38.793290"
}

--- Detailed extraction for Leon ---
{
  "name": "Leon",
  "source": "Pets in Turkey",
  "scraped_at": "2025-03-29T12:12:38.793402"
}

--- Detailed extraction for Orky ---
{
  "name": "Orky",
  "source": "Pets in Turkey",
  "scraped_at": "2025-03-29T12:12:38.793460"
}


In [28]:
# Extract data for all dogs
all_dogs_data = []

for name_element in dog_name_elements:
    dog_data = extract_dog_data(name_element)
    if dog_data:
        all_dogs_data.append(dog_data)

print(f"Successfully extracted data for {len(all_dogs_data)} dogs")

# Display the first 3 dogs as a sample
print("\nSample dog data (first 3 dogs):")
for i, dog in enumerate(all_dogs_data[:3]):
    print(f"\nDog {i+1}: {dog['name']}")
    for key, value in dog.items():
        if key != 'name' and key != 'images' and key != 'scraped_at':  # Skip images to keep output clean
            print(f"  {key}: {value}")
    if 'images' in dog:
        print(f"  images: {len(dog['images'])} images found")

All text elements in container:
0: I'm Norman
Successfully extracted data for 33 dogs

Sample dog data (first 3 dogs):

Dog 1: Norman
  source: Pets in Turkey

Dog 2: Leon
  source: Pets in Turkey

Dog 3: Orky
  source: Pets in Turkey


In [29]:
# Let's try a pattern-based approach by looking at the page as a whole
# This approach searches for patterns in the entire page rather than relying on HTML structure

def extract_dogs_from_page(soup):
    """Extract dog information using pattern matching in the entire page"""
    
    # Get all text from the page
    all_text = [t.strip() for t in soup.find_all(string=True) if t.strip()]
    
    # Find all dog name indices (where "I'm [Name]" appears)
    dog_indices = []
    for i, text in enumerate(all_text):
        if text.startswith("I'm "):
            dog_indices.append((i, text.replace("I'm ", "")))
    
    # Process each dog
    dogs_data = []
    for idx, (name_idx, name) in enumerate(dog_indices):
        # Determine the end of this dog's section
        end_idx = dog_indices[idx+1][0] if idx < len(dog_indices)-1 else len(all_text)
        
        # Extract the text segment for this dog
        dog_text = all_text[name_idx:end_idx]
        
        # Initialize dog data
        dog_data = {
            'name': name,
            'source': 'Pets in Turkey',
            'scraped_at': datetime.now().isoformat()
        }
        
        # Print the text for the first dog to help with debugging
        if name == "Norman":
            print(f"Text for {name}:")
            for i, text in enumerate(dog_text):
                print(f"{i}: {text}")
        
        # Extract status ("Ready to fly")
        for text in dog_text:
            if "Ready to fly" in text or "Currently in" in text:
                dog_data['status'] = text
                break
        
        # Look for fields and their values
        fields = ['Breed', 'Weight', 'Age', 'Sex', 'Neutered', 'Spayed']
        for field in fields:
            try:
                field_idx = dog_text.index(field)
                # Check if there's a value after this field
                if field_idx + 1 < len(dog_text):
                    value = dog_text[field_idx + 1]
                    # Make sure we're not capturing another field
                    if value not in fields:
                        dog_data[field.lower()] = value
            except ValueError:
                pass  # Field not found
        
        # Try to find height info (often combined with weight)
        for text in dog_text:
            if "height" in text.lower():
                if 'height' not in dog_data:
                    dog_data['height'] = text
                
                # If weight and height are combined, separate them
                if 'weight' in dog_data and dog_data['weight'] in text:
                    parts = text.split('height')
                    if len(parts) > 1:
                        dog_data['height'] = f"height{parts[1]}"
        
        dogs_data.append(dog_data)
    
    return dogs_data

# Get the dogs using our new method
dogs_data = extract_dogs_from_page(soup)

print(f"Successfully extracted data for {len(dogs_data)} dogs")

# Display the first 3 dogs
print("\nSample dog data (first 3 dogs):")
for i, dog in enumerate(dogs_data[:3]):
    print(f"\nDog {i+1}: {dog['name']}")
    for key, value in dog.items():
        if key != 'name' and key != 'scraped_at':
            print(f"  {key}: {value}")

Text for Norman:
0: I'm Norman
1: /$
2: $
3: xml version="1.0" encoding="UTF-8"?
4: /$
5: $
6: xml version="1.0" encoding="UTF-8"?
7: /$
8: $
9: xml version="1.0" encoding="UTF-8"?
10: /$
11: $
12: xml version="1.0" encoding="UTF-8"?
13: /$
14: $
15: xml version="1.0" encoding="UTF-8"?
16: /$
17: $
18: xml version="1.0" encoding="UTF-8"?
19: /$
20: $
21: xml version="1.0" encoding="UTF-8"?
22: /$
23: $
24: /$
25: $
26: xml version="1.0" encoding="UTF-8"?
27: /$
28: $
29: xml version="1.0" encoding="UTF-8"?
30: /$
31: $
32: Ready to fly!
33: /$
34: $
35: xml version="1.0" encoding="UTF-8"?
36: /$
37: $
38: xml version="1.0" encoding="UTF-8"?
39: /$
40: $
41: Breed
42: /$
43: $
44: Weight
45: /$
46: $
47: Age
48: /$
49: $
50: Sex
51: /$
52: $
53: Neutered
54: /$
55: $
56: Adopt Me
57: /$
58: $
59: Adopt Me
60: /$
61: $
62: Spaniel mix
63: /$
64: $
65: 20kg
66: height:49cm
67: /$
68: $
69: 2,5 yo
70: /$
71: $
72: Male
73: /$
74: $
75: Yes
76: /$
77: $
78: /$
79: /$
80: $
81: $
82: xml ver

In [30]:
def extract_dogs_from_page(soup):
    """Extract dog information using pattern matching in the entire page"""
    
    # Get all text from the page
    all_text = [t.strip() for t in soup.find_all(string=True) if t.strip()]
    
    # Filter out common noise patterns
    filtered_text = [t for t in all_text if t not in ["/$", "$", "/", "xml version=\"1.0\" encoding=\"UTF-8\"?"]]
    
    # Find all dog name indices (where "I'm [Name]" appears)
    dog_indices = []
    for i, text in enumerate(filtered_text):
        if text.startswith("I'm "):
            dog_indices.append((i, text.replace("I'm ", "")))
    
    # Process each dog
    dogs_data = []
    for idx, (name_idx, name) in enumerate(dog_indices):
        # Determine the end of this dog's section
        end_idx = dog_indices[idx+1][0] if idx < len(dog_indices)-1 else len(filtered_text)
        
        # Extract the text segment for this dog
        dog_text = filtered_text[name_idx:end_idx]
        
        # Initialize dog data
        dog_data = {
            'name': name,
            'source': 'Pets in Turkey',
            'scraped_at': datetime.now().isoformat()
        }
        
        # Print the text for the first dog to help with debugging
        if name == "Norman":
            print(f"Filtered text for {name}:")
            for i, text in enumerate(dog_text):
                print(f"{i}: {text}")
        
        # Extract status ("Ready to fly")
        for text in dog_text:
            if "Ready to fly" in text or "Currently in" in text:
                dog_data['status'] = text
                break
        
        # Look for fields and their values
        fields = ['Breed', 'Weight', 'Age', 'Sex', 'Neutered', 'Spayed']
        for field in fields:
            try:
                field_idx = dog_text.index(field)
                # Search for a value after this field that's not another field
                for i in range(field_idx + 1, min(field_idx + 5, len(dog_text))):
                    value = dog_text[i]
                    if value not in fields and not value.startswith("I'm ") and "xml" not in value:
                        # Found a valid value
                        dog_data[field.lower()] = value
                        break
            except ValueError:
                pass  # Field not found
        
        # Try to find height info (often combined with weight)
        for text in dog_text:
            if "height" in text.lower():
                dog_data['height'] = text
        
        dogs_data.append(dog_data)
    
    return dogs_data

# Get the dogs using our new method
dogs_data = extract_dogs_from_page(soup)

print(f"Successfully extracted data for {len(dogs_data)} dogs")

# Display the first 3 dogs
print("\nSample dog data (first 3 dogs):")
for i, dog in enumerate(dogs_data[:3]):
    print(f"\nDog {i+1}: {dog['name']}")
    for key, value in dog.items():
        if key != 'name' and key != 'scraped_at':
            print(f"  {key}: {value}")

Filtered text for Norman:
0: I'm Norman
1: Ready to fly!
2: Breed
3: Weight
4: Age
5: Sex
6: Neutered
7: Adopt Me
8: Adopt Me
9: Spaniel mix
10: 20kg
11: height:49cm
12: 2,5 yo
13: Male
14: Yes
Successfully extracted data for 33 dogs

Sample dog data (first 3 dogs):

Dog 1: Norman
  source: Pets in Turkey
  status: Ready to fly!
  weight: Adopt Me
  age: Adopt Me
  sex: Adopt Me
  neutered: Adopt Me
  height: height:49cm

Dog 2: Leon
  source: Pets in Turkey
  status: Ready to fly after 11/03/25
  weight: Adopt Me
  age: Adopt Me
  sex: Adopt Me
  neutered: Adopt Me
  height: height: 44cm

Dog 3: Orky
  source: Pets in Turkey
  status: Ready to fly  in June 2025
  weight: Adopt Me
  age: Adopt Me
  sex: Adopt Me
  neutered: Adopt Me
  height: height: 56cm


In [31]:
def extract_dogs_from_page(soup):
    """Extract dog information using pattern matching in the entire page"""
    
    # Get all text from the page
    all_text = [t.strip() for t in soup.find_all(string=True) if t.strip()]
    
    # Filter out common noise patterns
    filtered_text = [t for t in all_text if t not in ["/$", "$", "/", "xml version=\"1.0\" encoding=\"UTF-8\"?"]]
    
    # Find all dog name indices (where "I'm [Name]" appears)
    dog_indices = []
    for i, text in enumerate(filtered_text):
        if text.startswith("I'm "):
            dog_indices.append((i, text.replace("I'm ", "")))
    
    # Process each dog
    dogs_data = []
    for idx, (name_idx, name) in enumerate(dog_indices):
        # Determine the end of this dog's section
        end_idx = dog_indices[idx+1][0] if idx < len(dog_indices)-1 else len(filtered_text)
        
        # Extract the text segment for this dog
        dog_text = filtered_text[name_idx:end_idx]
        
        # Initialize dog data
        dog_data = {
            'name': name,
            'source': 'Pets in Turkey',
            'scraped_at': datetime.now().isoformat()
        }
        
        # Print the text for the first dog to help with debugging
        if name == "Norman":
            print(f"Filtered text for {name}:")
            for i, text in enumerate(dog_text):
                print(f"{i}: {text}")
        
        # Extract status ("Ready to fly")
        for text in dog_text:
            if "Ready to fly" in text or "Currently in" in text:
                dog_data['status'] = text
                break
        
        # Now we know from the filtered text that the structured data appears in a specific order:
        # Breed, Weight, Age, Sex, Neutered/Spayed followed by the corresponding values
        
        # Based on Norman's example, the field values appear to be at these positions:
        field_to_value = {
            'Breed': 9,  # "Spaniel mix"
            'Weight': 10,  # "20kg"
            'Age': 12,    # "2,5 yo"
            'Sex': 13,    # "Male"
            'Neutered': 14, # "Yes"
            'Spayed': 14   # Same as Neutered
        }
        
        # For each field, check if it exists in the dog text and get its value
        fields = ['Breed', 'Weight', 'Age', 'Sex', 'Neutered', 'Spayed']
        for field in fields:
            try:
                field_idx = dog_text.index(field)
                # Check if we have a mapping for this field
                if field in field_to_value:
                    value_idx = name_idx + field_to_value[field]
                    if value_idx < len(filtered_text):
                        value = filtered_text[value_idx]
                        # Skip "Adopt Me" or other obvious non-values
                        if value != "Adopt Me" and not value.startswith("I'm "):
                            dog_data[field.lower()] = value
            except ValueError:
                pass  # Field not found
        
        # Try to find height info (often combined with weight)
        for text in dog_text:
            if "height" in text.lower():
                dog_data['height'] = text
        
        dogs_data.append(dog_data)
    
    return dogs_data

# Get the dogs using our new method
dogs_data = extract_dogs_from_page(soup)

print(f"Successfully extracted data for {len(dogs_data)} dogs")

# Display the first 3 dogs
print("\nSample dog data (first 3 dogs):")
for i, dog in enumerate(dogs_data[:3]):
    print(f"\nDog {i+1}: {dog['name']}")
    for key, value in dog.items():
        if key != 'name' and key != 'scraped_at':
            print(f"  {key}: {value}")

Filtered text for Norman:
0: I'm Norman
1: Ready to fly!
2: Breed
3: Weight
4: Age
5: Sex
6: Neutered
7: Adopt Me
8: Adopt Me
9: Spaniel mix
10: 20kg
11: height:49cm
12: 2,5 yo
13: Male
14: Yes
Successfully extracted data for 33 dogs

Sample dog data (first 3 dogs):

Dog 1: Norman
  source: Pets in Turkey
  status: Ready to fly!
  breed: Spaniel mix
  weight: 20kg
  age: 2,5 yo
  sex: Male
  neutered: Yes
  height: height:49cm

Dog 2: Leon
  source: Pets in Turkey
  status: Ready to fly after 11/03/25
  breed: Terrier
  weight: 11kg
  age: 9 mnths old
  sex: Male
  neutered: Yes
  height: height: 44cm

Dog 3: Orky
  source: Pets in Turkey
  status: Ready to fly  in June 2025
  breed: Golden Retriever
  weight: 35kg
  age: 1,5 y/o
  sex: Male
  neutered: Yes
  height: height: 56cm


In [32]:
def extract_dogs_from_page(soup):
    """Extract dog information using a more position-based approach"""
    
    # Get all text from the page
    all_text = [t.strip() for t in soup.find_all(string=True) if t.strip()]
    
    # Filter out common noise patterns
    filtered_text = [t for t in all_text if t not in ["/$", "$", "/", "xml version=\"1.0\" encoding=\"UTF-8\"?"]]
    
    # We know the dog information is structured in a pattern:
    # Looking at Norman's example, we have:
    # "I'm Norman", "Ready to fly!", "Breed", ..., "Spaniel mix", "20kg", "2,5 yo", "Male", "Yes"
    
    # Find all dog name indices (where "I'm [Name]" appears)
    dogs_data = []
    for i, text in enumerate(filtered_text):
        if text.startswith("I'm "):
            name = text.replace("I'm ", "")
            
            # Initialize dog data
            dog_data = {
                'name': name,
                'source': 'Pets in Turkey',
                'scraped_at': datetime.now().isoformat()
            }
            
            # Look for status within the next 10 elements
            for j in range(i+1, min(i+10, len(filtered_text))):
                if "Ready to fly" in filtered_text[j] or "Currently in" in filtered_text[j]:
                    dog_data['status'] = filtered_text[j]
                    break
            
            # Look for "Breed" field
            try:
                breed_idx = filtered_text.index("Breed", i, i+15)
                # Based on Norman's example, we have these indices:
                # Breed: 2, Value: 9 (diff: 7)
                # Weight: 3, Value: 10 (diff: 7)
                # Age: 4, Value: 12 (diff: 8)
                # Sex: 5, Value: 13 (diff: 8)
                # Neutered: 6, Value: 14 (diff: 8)
                
                # Map field positions to value positions
                field_map = {
                    "Breed": 7,
                    "Weight": 7,
                    "Age": 8,
                    "Sex": 8,
                    "Neutered": 8,
                    "Spayed": 8
                }
                
                # Find all field positions
                for field, offset in field_map.items():
                    try:
                        field_pos = filtered_text.index(field, i, i+20)
                        value_pos = field_pos + offset
                        if value_pos < len(filtered_text):
                            value = filtered_text[value_pos]
                            # Avoid using "Adopt Me" as a value
                            if value != "Adopt Me" and not value.startswith("I'm "):
                                dog_data[field.lower()] = value
                    except ValueError:
                        pass
                
                # Try to find height info
                for j in range(i, min(i+20, len(filtered_text))):
                    if "height" in filtered_text[j].lower():
                        dog_data['height'] = filtered_text[j]
                        break
                
                # Print details for Norman to debug
                if name == "Norman":
                    print(f"Extraction details for {name}:")
                    print(f"Position in filtered_text: {i}")
                    for j in range(i, min(i+20, len(filtered_text))):
                        print(f"{j-i}: {filtered_text[j]}")
                    print("Extracted fields:")
                    for key, value in dog_data.items():
                        if key not in ['name', 'source', 'scraped_at']:
                            print(f"  {key}: {value}")
                
                dogs_data.append(dog_data)
            except ValueError:
                # No "Breed" field found, skip this dog
                continue
    
    return dogs_data

# Test our new extraction approach
new_dogs_data = extract_dogs_from_page(soup)

print(f"Successfully extracted data for {len(new_dogs_data)} dogs")

# Display the first 3 dogs
print("\nSample dog data (first 3 dogs):")
for i, dog in enumerate(new_dogs_data[:3]):
    print(f"\nDog {i+1}: {dog['name']}")
    for key, value in dog.items():
        if key != 'name' and key != 'scraped_at' and key != 'source':
            print(f"  {key}: {value}")

Extraction details for Norman:
Position in filtered_text: 68
0: I'm Norman
1: Ready to fly!
2: Breed
3: Weight
4: Age
5: Sex
6: Neutered
7: Adopt Me
8: Adopt Me
9: Spaniel mix
10: 20kg
11: height:49cm
12: 2,5 yo
13: Male
14: Yes
15: I'm Leon
16: Ready to fly after 11/03/25
17: Breed
18: Weight
19: Age
Extracted fields:
  status: Ready to fly!
  breed: Spaniel mix
  weight: 20kg
  age: 2,5 yo
  sex: Male
  neutered: Yes
  height: height:49cm
Successfully extracted data for 33 dogs

Sample dog data (first 3 dogs):

Dog 1: Norman
  status: Ready to fly!
  breed: Spaniel mix
  weight: 20kg
  age: 2,5 yo
  sex: Male
  neutered: Yes
  height: height:49cm

Dog 2: Leon
  status: Ready to fly after 11/03/25
  breed: Terrier
  weight: 11kg
  age: 9 mnths old
  sex: Male
  neutered: Yes
  height: height: 44cm

Dog 3: Orky
  status: Ready to fly  in June 2025
  breed: Golden Retriever
  weight: 35kg
  age: 1,5 y/o
  sex: Male
  neutered: Yes
  height: height: 56cm


In [33]:
# Focus on extracting dog images
def extract_dog_images(soup):
    """Extract images for each dog"""
    
    # Find all dog name elements
    dog_name_elements = soup.find_all(string=lambda t: t and "I'm " in t)
    dog_images = {}
    
    for name_elem in dog_name_elements:
        dog_name = name_elem.strip().replace("I'm ", "")
        
        # Find the container for this dog
        container = name_elem.parent
        while container and container.name != 'div':
            container = container.parent
        
        if not container:
            continue
        
        # Find all images in this container
        images = container.find_all('img')
        image_urls = []
        
        for img in images:
            src = img.get('src', '')
            if src and not src.startswith('data:'):  # Skip base64 encoded images
                # Add domain if it's a relative URL
                if not src.startswith('http'):
                    src = base_url + ('' if src.startswith('/') else '/') + src
                image_urls.append(src)
        
        if image_urls:
            dog_images[dog_name] = image_urls
    
    return dog_images

# Try to extract images
dog_images = extract_dog_images(soup)

# Check if we found any images
print(f"Found images for {len(dog_images)} dogs")

# Look at a few examples
for name in list(dog_images.keys())[:3]:
    print(f"\nImages for {name}:")
    for i, url in enumerate(dog_images[name]):
        print(f"  Image {i+1}: {url}")

Found images for 0 dogs
